# Лекция: Статистические критерии в Python

**Дисциплина:** Введение в анализ больших данных  
**Задание 10** (адаптация с языка R на Python)

Темы:
- Shapiro–Wilk (нормальность)
- Критерий Фишера (равенство дисперсий)
- Kolmogorov–Smirnov (одна и две выборки)
- t-тест Стьюдента
- Критерий Пирсона (хи-квадрат) для таблиц сопряжённости
- Мозаичные диаграммы

Библиотеки: **scipy.stats**, **pandas**, **statsmodels**, **matplotlib**.


In [ ]:
import numpy as np
import pandas as pd
from scipy import stats
import matplotlib.pyplot as plt
import seaborn as sns

plt.rcParams['figure.figsize'] = (10, 6)
sns.set_style("whitegrid")
np.random.seed(42)
print("Библиотеки загружены")


---
## 1. Датасет `trees`

Замеры диаметра (Girth), высоты (Height) и объёма (Volume) вишнёвых деревьев.


In [ ]:
url = "https://vincentarelbundock.github.io/Rdatasets/csv/datasets/trees.csv"
trees = pd.read_csv(url, index_col=0)
print("Имена столбцов:", trees.columns.tolist())
print(trees.head())
print("\nРазмерность:", trees.shape)


### Shapiro–Wilk: проверка нормальности

В R: `shapiro.test(x)`  
H0: данные распределены нормально. При p > 0.05 гипотезу **не отвергаем**.


In [ ]:
print("=== Shapiro–Wilk (trees) ===")
for col in trees.columns:
    W, p = stats.shapiro(trees[col])
    verdict = "нормальное (не отвергаем H0)" if p > 0.05 else "НЕ нормальное (отвергаем H0)"
    print(f"{col:8s}: W={W:.4f}, p={p:.4f}  →  {verdict}")


### Критерий Фишера: равенство дисперсий

В R: `var.test(x, y)`  
H0: дисперсии двух выборок равны.


In [ ]:
cols = trees.columns.tolist()
print("=== F-тест (равенство дисперсий) ===")
for i, c1 in enumerate(cols):
    for c2 in cols[i+1:]:
        F = trees[c1].var(ddof=1) / trees[c2].var(ddof=1)
        dfn, dfd = len(trees[c1]) - 1, len(trees[c2]) - 1
        p = 2 * min(stats.f.cdf(F, dfn, dfd), 1 - stats.f.cdf(F, dfn, dfd))
        verdict = "дисперсии равны" if p > 0.05 else "дисперсии РАЗЛИЧАЮТСЯ"
        print(f"{c1:8s} vs {c2:8s}: F={F:.4f}, p={p:.4f}  →  {verdict}")


---
## 2. Датасет `randu`

400 троек псевдослучайных чисел из [0, 1] (столбцы x, y, z).


In [ ]:
url_randu = "https://vincentarelbundock.github.io/Rdatasets/csv/datasets/randu.csv"
randu = pd.read_csv(url_randu, index_col=0)
print(randu.head())
print("Размерность:", randu.shape)
print(randu.describe().round(3))


### Двувыборочный Kolmogorov–Smirnov

В R: `ks.test(x, y)`  
H0: две выборки из одного непрерывного распределения.


In [ ]:
pairs = [("x", "y"), ("x", "z"), ("y", "z")]
print("=== KS two-sample (randu) ===")
for a, b in pairs:
    D, p = stats.ks_2samp(randu[a], randu[b])
    verdict = "одно распределение" if p > 0.05 else "РАЗНЫЕ распределения"
    print(f"{a} vs {b}: D={D:.4f}, p={p:.4f}  →  {verdict}")


### Одновыборочный KS: согласие с заданным распределением

В R: `ks.test(x, "pnorm")`, `ks.test(y, "punif")`


In [ ]:
mu, sigma = randu["x"].mean(), randu["x"].std(ddof=1)
D, p = stats.kstest(randu["x"], "norm", args=(mu, sigma))
print(f"x ~ Normal({mu:.3f}, {sigma:.3f}): D={D:.4f}, p={p:.4f}")
print("  →", "не отвергаем нормальность" if p > 0.05 else "отвергаем нормальность")

D, p = stats.kstest(randu["y"], "uniform", args=(0, 1))
print(f"\ny ~ Uniform(0, 1): D={D:.4f}, p={p:.4f}")
print("  →", "не отвергаем равномерность" if p > 0.05 else "отвергаем равномерность")


---
## 3. Временной ряд `ldeaths`

Ежемесячная смертность от болезней лёгких в UK (1974–1979).


In [ ]:
url_ld = "https://vincentarelbundock.github.io/Rdatasets/csv/datasets/ldeaths.csv"
ld = pd.read_csv(url_ld)
print(ld.head(12))
print("Столбцы:", ld.columns.tolist())


In [ ]:
values = ld.iloc[:, -1].values
years = np.repeat(np.arange(1974, 1980), 12)[:len(values)]
months = np.tile(np.arange(1, 13), 6)[:len(values)]

ldeaths = pd.DataFrame({"year": years, "month": months, "deaths": values})
print(ldeaths.groupby("year")["deaths"].agg(["mean", "std", "count"]).round(1))


### t-тест: равенство средних по годам

В R: `t.test(x, y)`


In [ ]:
print("=== t-тест средних смертности по годам ===")
year_list = sorted(ldeaths["year"].unique())
for i, y1 in enumerate(year_list):
    for y2 in year_list[i+1:]:
        a = ldeaths.loc[ldeaths["year"] == y1, "deaths"]
        b = ldeaths.loc[ldeaths["year"] == y2, "deaths"]
        t, p = stats.ttest_ind(a, b, equal_var=False)
        verdict = "средние равны" if p > 0.05 else "средние РАЗЛИЧАЮТСЯ"
        print(f"{y1} vs {y2}: t={t:7.3f}, p={p:.4f}  →  {verdict}")


### F-тест: равенство дисперсий по годам


In [ ]:
print("=== F-тест дисперсий по годам ===")
for i, y1 in enumerate(year_list):
    for y2 in year_list[i+1:]:
        a = ldeaths.loc[ldeaths["year"] == y1, "deaths"]
        b = ldeaths.loc[ldeaths["year"] == y2, "deaths"]
        F = a.var(ddof=1) / b.var(ddof=1)
        dfn, dfd = len(a) - 1, len(b) - 1
        p = 2 * min(stats.f.cdf(F, dfn, dfd), 1 - stats.f.cdf(F, dfn, dfd))
        verdict = "дисперсии равны" if p > 0.05 else "дисперсии РАЗЛИЧАЮТСЯ"
        print(f"{y1} vs {y2}: F={F:.3f}, p={p:.4f}  →  {verdict}")


---
## 4. `HairEyeColor` — цвет волос и глаз

Таблица сопряжённости 592 студентов.  
H0 (Пирсон): цвет глаз **не зависит** от цвета волос.


In [ ]:
url_he = "https://vincentarelbundock.github.io/Rdatasets/csv/datasets/HairEyeColor.csv"
he = pd.read_csv(url_he, index_col=0)
print(he.head(10))
print("\nСтолбцы:", he.columns.tolist())


In [ ]:
def make_table(df, sex):
    sub = df[df["Sex"] == sex]
    tab = sub.pivot_table(index="Hair", columns="Eye", values="Freq", aggfunc="sum", fill_value=0)
    return tab

male = make_table(he, "Male")
female = make_table(he, "Female")
print("Male:\n", male)
print("\nFemale:\n", female)


### Критерий Пирсона (хи-квадрат)

В R: `chisq.test(table)`


In [ ]:
def chi2_independence(table, name):
    chi2, p, dof, expected = stats.chi2_contingency(table)
    print(f"=== {name} ===")
    print(f"chi2 = {chi2:.3f}, df = {dof}, p = {p:.4e}")
    if p < 0.05:
        print("  → отвергаем H0: цвет глаз ЗАВИСИТ от цвета волос")
    else:
        print("  → не отвергаем H0: независимость")
    print()
    return chi2, p

chi2_independence(male, "Male")
chi2_independence(female, "Female")


### Мозаичные диаграммы

В R: `mosaicplot(...)`  
В Python: `statsmodels.graphics.mosaicplot.mosaic`


In [ ]:
from statsmodels.graphics.mosaicplot import mosaic

fig, axes = plt.subplots(1, 2, figsize=(14, 6))

def to_mosaic_data(table):
    d = {}
    for hair in table.index:
        for eye in table.columns:
            d[(str(hair), str(eye))] = table.loc[hair, eye]
    return d

mosaic(to_mosaic_data(male), ax=axes[0], title="Male: Hair vs Eye",
       properties=lambda k: {"color": "steelblue"})
mosaic(to_mosaic_data(female), ax=axes[1], title="Female: Hair vs Eye",
       properties=lambda k: {"color": "coral"})
plt.tight_layout()
plt.show()


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 5))
sns.heatmap(male, annot=True, fmt=".0f", cmap="Blues", ax=axes[0])
axes[0].set_title("Male")
sns.heatmap(female, annot=True, fmt=".0f", cmap="Oranges", ax=axes[1])
axes[1].set_title("Female")
plt.suptitle("Частоты: цвет волос × цвет глаз", y=1.02)
plt.tight_layout()
plt.show()


---
## Шпаргалка: R → Python

| Задача в R | Python |
|------------|--------|
| `shapiro.test(x)` | `stats.shapiro(x)` |
| `ks.test(x, y)` | `stats.ks_2samp(x, y)` |
| `ks.test(x, "pnorm")` | `stats.kstest(x, "norm", args=(mu, sigma))` |
| `ks.test(x, "punif")` | `stats.kstest(x, "uniform", args=(0, 1))` |
| `t.test(x, y)` | `stats.ttest_ind(x, y)` |
| `var.test(x, y)` | F = var1/var2 + `stats.f.cdf` |
| `chisq.test(table)` | `stats.chi2_contingency(table)` |
| `mosaicplot(table)` | `statsmodels.graphics.mosaicplot.mosaic` |

---
## Рекомендации

1. При p > 0.05 обычно **не отвергаем** H0 (на уровне 0.05).
2. Shapiro–Wilk надёжен при n ≤ 5000.
3. Для KS с нормальным распределением параметры лучше оценивать по выборке.
4. Хи-квадрат требует достаточных ожидаемых частот (обычно ≥ 5).

**Удачи с выполнением Задания 10!**
